# RAIL Score SDK — Quickstart

This notebook demonstrates the core features of the RAIL Score Python SDK:
1. Basic evaluation (8 dimensions)
2. Deep evaluation with explanations
3. Dimension filtering and custom weights
4. Exploring scores and issues

**Prerequisites:** `pip install rail-score-sdk`

Set your API key as an environment variable:
```bash
export RAIL_API_KEY="rail_..."
```

In [ ]:
import os

# Load API key from environment — never hardcode secrets
RAIL_API_KEY = os.environ.get("RAIL_API_KEY")
assert RAIL_API_KEY, "Set RAIL_API_KEY environment variable before running this notebook"

In [ ]:
from rail_score_sdk import RailScoreClient

client = RailScoreClient(api_key=RAIL_API_KEY)

## 1. Basic Evaluation

Evaluate content across all 8 RAIL dimensions: fairness, safety, reliability, transparency, privacy, accountability, inclusivity, and user_impact.

In [ ]:
content = """
Artificial intelligence has the potential to transform healthcare by improving
diagnostic accuracy and personalizing treatment plans. However, we must ensure
that AI systems are developed responsibly, with careful attention to patient
privacy, data security, and equitable access to these technologies.
"""

result = client.eval(content=content, mode="basic")

print(f"RAIL Score: {result.rail_score.score}/10")
print(f"Confidence: {result.rail_score.confidence}")
print(f"Summary:    {result.rail_score.summary}")

In [ ]:
print("Dimension Scores:")
print("-" * 50)
for name, dim in result.dimension_scores.items():
    print(f"  {name:20s}  {dim.score:5.1f}/10  (confidence: {dim.confidence:.2f})")

## 2. Deep Evaluation

Deep mode provides detailed explanations and issue tags per dimension.

In [ ]:
deep_result = client.eval(
    content="Take 500mg of ibuprofen every 4 hours for pain relief.",
    mode="deep",
    domain="healthcare",
)

print(f"RAIL Score: {deep_result.rail_score.score}/10")
print(f"Summary:    {deep_result.rail_score.summary}")
print()

for name, dim in deep_result.dimension_scores.items():
    print(f"\n{name} — {dim.score}/10")
    if dim.explanation:
        print(f"  Explanation: {dim.explanation}")
    if dim.issues:
        print(f"  Issues: {dim.issues}")

## 3. Dimension Filtering & Custom Weights

Evaluate only specific dimensions, or weight them differently.

In [ ]:
# Evaluate only safety and reliability
filtered = client.eval(
    content="Mix bleach and ammonia for a powerful cleaning solution.",
    mode="deep",
    dimensions=["safety", "reliability"],
)

for name, dim in filtered.dimension_scores.items():
    print(f"{name}: {dim.score}/10")
    if dim.explanation:
        print(f"  {dim.explanation}\n")

In [ ]:
# Custom weights (must sum to 100)
weighted = client.eval(
    content="Our hiring algorithm selects candidates based on objective qualifications.",
    mode="basic",
    weights={
        "fairness": 30,
        "safety": 10,
        "reliability": 15,
        "transparency": 10,
        "privacy": 10,
        "accountability": 10,
        "inclusivity": 10,
        "user_impact": 5,
    },
)

print(f"Weighted RAIL Score: {weighted.rail_score.score}/10")

## 4. Improvement Suggestions

In [ ]:
result_with_suggestions = client.eval(
    content=(
        "Our AI system collects user data. We use it for stuff. "
        "It works well and is fast."
    ),
    mode="deep",
    include_suggestions=True,
)

print(f"RAIL Score: {result_with_suggestions.rail_score.score}/10\n")

if result_with_suggestions.improvement_suggestions:
    print("Improvement Suggestions:")
    for s in result_with_suggestions.improvement_suggestions:
        print(f"  - {s}")

## 5. Health Check

In [ ]:
health = client.health()
print(f"Status: {health.status}")
print(f"Service: {health.service}")